# 11 — Word Embeddings via LSA
**Goal:** Obtener vectores densos de palabras que capturen semántica — sin Word2Vec ni transformers.

## Latent Semantic Analysis (LSA)

$$M_{\text{TF-IDF}} \xrightarrow{\text{SVD}} U \Sigma V^T$$

- $M$: document-term matrix  (docs × vocab)
- $V$: **word embeddings** — cada fila es un vector denso de una palabra
- $U\Sigma$: **document embeddings** — cada fila es un vector denso de un doc

La magia: palabras que aparecen en contextos similares tendrán vectores similares aunque nunca co-ocurran exactamente.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

STOPWORDS_ES = [
    'a','al','como','con','de','del','el','en','es','la','las','lo','los',
    'me','mi','muy','no','o','para','por','que','se','si','sin','su','te',
    'todo','un','una','y','yo','tuve','llevo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
# Corpus más variado para que el embedding capture más semántica
corpus = [
    # app / velocidad
    'La app está muy lenta y tarda en cargar',
    'La aplicación carga lento, no puedo entrar',
    'La app es rápida y responde de inmediato',
    'La aplicación es rápida y muy fluida',
    'El sistema es lento y tarda demasiado',
    'La velocidad de la app es excelente',
    # OTP / seguridad
    'No me llegó el OTP al celular',
    'El código OTP no llega por SMS',
    'El SMS de verificación no aparece',
    'El OTP llegó al instante, todo bien',
    'El código de seguridad llegó rápido',
    'No recibo el código de verificación',
    # crédito / aprobación
    'Aprobaron mi crédito en minutos',
    'Mi solicitud de crédito fue aprobada rápido',
    'Rechazaron mi solicitud de tarjeta sin explicación',
    'La evaluación de crédito lleva días',
    'Mi aprobación de tarjeta fue inmediata',
    'El proceso de aprobación es muy lento',
    # soporte / atención
    'El soporte al cliente es excelente',
    'La atención al cliente es muy buena',
    'El servicio de soporte no responde',
    'El servicio al cliente es pésimo',
    'El agente de soporte resolvió mi problema',
    'Nadie del servicio al cliente me ayuda',
    # documentos
    'Subir documentos fue muy fácil',
    'La carga de documentos es sencilla',
    'No puedo subir mis documentos al sistema',
    'La carga de archivos falla constantemente',
    'El proceso de subir documentos es simple',
    'Los archivos no se suben correctamente',
    # positivo general
    'Excelente experiencia, recomiendo la tarjeta',
    'Increíbles beneficios, muy satisfecho',
    'Pésima experiencia, no recomiendo para nada',
    'Muy mala experiencia, no volvería a usar',
    'La tarjeta tiene beneficios increíbles',
    'Los beneficios de la tarjeta son malos',
] * 8   # repetir para más señal

print(f'Corpus: {len(corpus)} documentos')

## 1. Construir la document-term matrix y aplicar SVD

In [ ]:
# TF-IDF sobre el corpus
tfidf = TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES,
                         min_df=2, sublinear_tf=True)
M = tfidf.fit_transform(corpus)   # (docs, vocab) — sparse
feature_names = tfidf.get_feature_names_out()
print(f'M shape: {M.shape}  (docs × vocab)')

# SVD: M ≈ U × Σ × V^T
# V.T columns = word embeddings; U × Σ rows = document embeddings
N_COMPONENTS = 30
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
doc_embeddings = svd.fit_transform(M)            # U×Σ  (docs, 30)
word_embeddings = svd.components_.T               # V    (vocab, 30)

print(f'doc_embeddings shape:  {doc_embeddings.shape}')
print(f'word_embeddings shape: {word_embeddings.shape}')
print(f'Varianza explicada: {svd.explained_variance_ratio_.sum():.1%}')

## 2. Buscar palabras semánticamente similares

In [ ]:
word2idx = {w: i for i, w in enumerate(feature_names)}

def most_similar_words(word, top_k=8):
    if word not in word2idx:
        print(f'  "{word}" no está en el vocabulario')
        return
    idx = word2idx[word]
    vec = word_embeddings[idx].reshape(1, -1)                      # (1, 30)
    sims = cosine_similarity(vec, word_embeddings).ravel()         # (vocab,)
    top  = np.argsort(sims)[::-1][1:top_k+1]                      # excluir la palabra misma
    print(f'Palabras similares a "{word}":')
    for i in top:
        print(f'  {feature_names[i]:20s} {sims[i]:.3f}')

most_similar_words('lenta')
print()
most_similar_words('excelente')
print()
most_similar_words('soporte')

## 3. Analogías vectoriales — `lenta - malo + bueno ≈ rápida`

In [ ]:
def analogy(pos1, pos2, neg1, top_k=5):
    """
    pos1 es a pos2 como neg1 es a ???
    vector = emb(pos2) - emb(pos1) + emb(neg1)
    """
    exclude = {pos1, pos2, neg1}
    words = [pos1, pos2, neg1]
    for w in words:
        if w not in word2idx:
            print(f'  "{w}" no en vocabulario'); return

    vec = (word_embeddings[word2idx[pos2]]
           - word_embeddings[word2idx[pos1]]
           + word_embeddings[word2idx[neg1]])
    vec = vec.reshape(1, -1)
    sims = cosine_similarity(vec, word_embeddings).ravel()

    print(f'"{pos1}" → "{pos2}"  como  "{neg1}" → ???')
    shown = 0
    for i in np.argsort(sims)[::-1]:
        if feature_names[i] not in exclude:
            print(f'  {feature_names[i]:20s} {sims[i]:.3f}')
            shown += 1
            if shown >= top_k:
                break

# Con corpus pequeño las analogías son aproximadas
# pero la mecánica es la misma que Word2Vec / GloVe
analogy('lenta', 'rapida', 'mala')
print()
analogy('soporte', 'excelente', 'app')

## 4. Document embeddings — representar documentos en espacio semántico

In [ ]:
# Embedding de un documento nuevo = transform con el SVD ya entrenado
nuevos = [
    'La aplicación tarda mucho en responder',   # similar a "app lenta"
    'El soporte me ayudó rápidamente',           # similar a "soporte positivo"
    'No recibo el mensaje con el código',        # similar a "OTP"
]

X_new = tfidf.transform(nuevos)
emb_new = svd.transform(X_new)     # (3, 30)
emb_new_norm = Normalizer().fit_transform(emb_new)

# Comparar contra embeddings del corpus
doc_norm = Normalizer().fit_transform(doc_embeddings)

# Encontrar el doc más similar del corpus para cada query
sims = cosine_similarity(emb_new_norm, doc_norm)

print('Doc más similar del corpus a cada query:')
for i, query in enumerate(nuevos):
    best = np.argmax(sims[i])
    print(f'  Query: "{query}"')
    print(f'  Match: "{corpus[best]}"  (sim={sims[i, best]:.3f})\n')

## 5. Visualización — mapa semántico de palabras

In [ ]:
# Proyectar word embeddings a 2D
svd2 = TruncatedSVD(n_components=2, random_state=42)
word_2d = svd2.fit_transform(word_embeddings)   # (vocab, 2)

# Palabras de interés
words_to_plot = [
    'lenta', 'rapida', 'lento', 'veloz',
    'excelente', 'pesimo', 'buena', 'mala',
    'soporte', 'atencion', 'servicio',
    'otp', 'codigo', 'sms', 'verificacion',
    'credito', 'aprobacion', 'solicitud',
    'documentos', 'archivos', 'carga',
]

fig, ax = plt.subplots(figsize=(11, 8))
COLORS_CAT = {
    'veloc':  ('#4361ee', ['lenta', 'rapida', 'lento']),
    'valor':  ('#f72585', ['excelente', 'pesimo', 'buena', 'mala']),
    'sopor':  ('#06d6a0', ['soporte', 'atencion', 'servicio']),
    'otp':    ('#ff9f1c', ['otp', 'codigo', 'sms', 'verificacion']),
    'credit': ('#7209b7', ['credito', 'aprobacion', 'solicitud']),
    'docs':   ('#e63946', ['documentos', 'archivos', 'carga']),
}

for cat, (color, words) in COLORS_CAT.items():
    for w in words:
        if w in word2idx:
            idx = word2idx[w]
            x, y = word_2d[idx]
            ax.scatter(x, y, c=color, s=80, zorder=3)
            ax.annotate(w, (x, y), textcoords='offset points',
                       xytext=(5, 5), fontsize=9, color=color)

ax.set_title('Mapa semántico de palabras (LSA word embeddings → 2D)')
ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
plt.tight_layout()
plt.show()

## 6. LSA vs Word2Vec vs Transformers

| | LSA | Word2Vec / GloVe | BERT / transformers |
|---|---|---|---|
| Método | SVD sobre TF-IDF | Red neuronal shallow | Red profunda (atención) |
| Contexto | Estadístico global | Ventana local | Toda la oración |
| Polisemia | No — un vector por palabra | No — un vector por palabra | Sí — vector depende del contexto |
| Librerías | `sklearn` | `gensim`, `fasttext` | `transformers` (HuggingFace) |
| Tamaño corpus | Pequeño | Grande (100M+ tokens) | Muy grande (pre-entrenado) |
| Sin dependencias | ✅ | ❌ | ❌ |

## Resumen
| Concepto | API / fórmula |
|---|---|
| Word embeddings (LSA) | `svd.components_.T` → (vocab, n_components) |
| Document embeddings | `svd.fit_transform(X_tfidf)` → (docs, n_components) |
| Similitud semántica | `cosine_similarity(vec_a, word_embeddings)` |
| Analogía | `emb(b) - emb(a) + emb(c)` → buscar vecino |
| Embed nuevos docs | `svd.transform(tfidf.transform(new_docs))` |

**Siguiente:** `12_language_models.ipynb` — modelar la probabilidad del lenguaje con n-gramas.